In [1]:
import json
import sys
import os
import numpy as np
import random
from sklearn.model_selection import KFold
import torch
from torch.utils.data import DataLoader

models_path = os.path.abspath(os.path.join('..', 'models'))
sys.path.append(models_path)

models_path = os.path.abspath(os.path.join('..', 'src'))
sys.path.append(models_path)

from reindex import reindex_problem_ids
from collate_batch import collate_batch
from answer_set import AnswerSet
from DKT.dkt_evaluation import calculate_loss
from DKT.dkt_model import DKT
from DKT.dkt_process import process


In [ ]:

def return_assistments_dict_dkt(
        df_answers: pd.DataFrame
) -> Dict[str, List[Tuple[int, Literal[0, 1]]]]:
    """
    Returns the assistments data as a Deep Knowledge Tracing (DKT) dictionary.

    This function processes the data, filtering out problems and users that don't meet 
    the required frequency criteria, and generates a dictionary of answers for each user.
    
    Parameters:

    Returns:
    - dict: A dictionary where the keys are user IDs and the values are lists of tuples 
      (problem_id, correct) representing each user’s interactions with problems.
    """
    # Load the dataset
    # Create the desired dictionary where each user_id maps to a list of (problem_id, correct) tuples
    result_dict = defaultdict(list)

    # Group by 'user_id' and iterate through each group
    for user_id, user_group in df_answers.groupby('user_id'):
        result_dict[user_id] = list(zip(user_group['problem_id'], user_group['correct']))

    return result_dict

In [2]:
# HIGHEST_PROBLEM_ID = 12937

with open('..\\data\\preprocessed\\assistments_user_dict.json', 'r') as json_file:
    user_dict = json.load(json_file)

user_dict_reindexed, mapping_dict, num_problem_ids = reindex_problem_ids(user_dict)

output_file = '../data/preprocessed/assistments_user_dict_reindexed.json'
# Save the user dictionary to a JSON file
with open(output_file, 'w') as json_file:
    json.dump(user_dict_reindexed, json_file)
print(f"Reindexed user dictionary saved to {output_file}.")

print(f"Number of different problems: {num_problem_ids}")


Reindexed user dictionary saved to ../data/preprocessed/assistments_user_dict_reindexed.json.
Number of different problems: 12937


The number of unique exercises is too high, we need random vector representations.
We should transform to the following dimension:

In [3]:
embed_dim = round(np.log(num_problem_ids) + 1)
print(embed_dim)

10


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_EPOCHS = 15
BATCH_SIZE = 100
LEARNING_RATE = 1e-4
NUM_FOLDS = 5  # Number of folds for cross-validation

config = {
    "num_items": num_problem_ids,
    "embed_dim": embed_dim,
    "hid_size": 200,
    "num_hid_layers": 2,
    "drop_prob": 0.5
}


In [ ]:
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
keys = list(user_dict_reindexed.keys())
random.shuffle(keys)

best_val_auc = 0  # Track the best validation AUC

test_auc_list = []
fold_model_params_list = []

for fold, (train_idx, test_idx) in enumerate(kf.split(keys)):
    print(f"\nFold {fold+1}/{NUM_FOLDS}")
    train_keys = [keys[i] for i in train_idx]
    test_keys = [keys[i] for i in test_idx]

    train_dict = {key: user_dict_reindexed[key] for key in train_keys}
    test_dict = {key: user_dict_reindexed[key] for key in test_keys}

    train_dataset = AnswerSet(train_dict)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=collate_batch, pin_memory=True, num_workers=2)

    test_dataset = AnswerSet(test_dict)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_batch, pin_memory=True, num_workers=2)

    # Create model
    model = DKT(**config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_fold_val_auc = 0
    consecutive_epochs_no_improve = 0
    for epoch in range(1, NUM_EPOCHS+1):
        print(f"\n{epoch}. epoch\n")
        # Training
        process(model, train_loader, calculate_loss, device, optimizer)

        # Evaluate model on validation set
    test_loss, test_auc = process(model, test_loader, device, calculate_loss)

    test_auc_list.append(test_auc)
    fold_model_params_list.append(model.state_dict())

    # Update the global best if needed
    if test_auc > best_val_auc:
        best_val_auc = test_auc
        best_model_params = fold_model_params_list[fold]  # Store the best fold parameters
        print(f"\nNew best model found: test AUC: {best_val_auc:.4f}, fold: {fold}")

print(f"\nBest test AUC: {best_val_auc:.4f}")

# Save the best model parameters
torch.save(best_model_params, 'best_model.pt')



Fold 1/5

1. epoch

Training: 32 batches [00:05,  5.33 batches/s]

2. epoch

Training: 32 batches [00:03,  8.41 batches/s]

3. epoch

Training: 32 batches [00:03,  8.42 batches/s]

4. epoch

Training: 32 batches [00:04,  6.77 batches/s]

5. epoch

Training: 32 batches [00:03,  8.33 batches/s]

6. epoch

Training: 32 batches [00:03,  8.43 batches/s]

7. epoch

Training: 32 batches [00:04,  7.62 batches/s]

8. epoch

Training: 32 batches [00:04,  7.85 batches/s]

9. epoch

Training: 32 batches [00:03,  8.32 batches/s]

10. epoch

Training: 32 batches [00:04,  7.84 batches/s]

11. epoch

Training: 32 batches [00:04,  7.37 batches/s]

12. epoch

Training: 32 batches [00:03,  8.49 batches/s]

13. epoch

Training: 32 batches [00:04,  7.95 batches/s]

14. epoch

Training: 32 batches [00:04,  6.72 batches/s]

15. epoch

Training: 32 batches [00:03,  8.37 batches/s]
Evaluation: 8 batches [00:00,  9.72 batches/s]

New best model found: test AUC: 0.8189, fold: 0

Fold 2/5

1. epoch

Training: 32

In [17]:
print(f"Average achieved AUC: {sum(test_auc_list)/len(test_auc_list)}")

Average achieved AUC: 0.8170440389679147
